# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step workflow for loading, exploring, and processing the FAIR\$^2\$ dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL. This ensures that all metadata and file references are FAIR and machine-readable.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata  # Do not treat as dictionary for further access

print(f"{metadata_obj.name}\n{'='*len(metadata_obj.name)}\n{metadata_obj.description}\n")
print(f"Dataset identifier: {metadata_obj.identifier}")
print(f"Published: {metadata_obj.date_published}")
print(f"License: {metadata_obj.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id` keys for granular programmatic access.

Croissant identifies logical tables as `record sets`. Each record set contains fields (columns) and those are referenced by their unique `@id` identifier.

Let's enumerate all record set `@id` fields and view their basic structure.

In [ ]:
print("Available record sets and their fields:\n")

# Extract all record sets
record_sets = dataset.metadata.record_sets  # <--- This is a list of RecordSet objects
if not record_sets:
    print("No record sets found in the metadata. Please review dataset documentation.")
else:
    for rs in record_sets:
        print(f"- Record set name: {rs.name}")
        print(f"  @id: {rs.id}")
        fields = rs.fields
        print(f"  Fields:")
        for field in fields:
            print(f"      - {field.name} (@id={field.id}, dataType={field.data_type})")
        print()

Let's also enumerate the `@id` values which will be used to reference entities in later programmatic steps.

In [ ]:
# List record set ids and field ids for reference
record_set_ids = []
record_set_field_ids_map = {}
if record_sets:
    for rs in record_sets:
        record_set_ids.append(rs.id)
        field_ids = [field.id for field in rs.fields]
        record_set_field_ids_map[rs.id] = field_ids
print("Record set @ids:")
print(record_set_ids)
print("Field @ids per record set:")
for rs_id, field_ids in record_set_field_ids_map.items():
    print(f"{rs_id}: {field_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We'll load all records as DataFrames and inspect the main table(s).

In [ ]:
dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nLoading record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))  # Yields dicts
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from `{record_set_id}`.")
    print("Columns:", df.columns.tolist())
    print(df.head(2))
    print("-")

In the following cells, we will select one of the main record sets for further analysis. Below, replace `<main_record_set_id>` with the appropriate `@id` from those printed above.

In [ ]:
# Choose the main data record set for downstream steps:
# e.g., main_record_set_id = 'cr:RecordSet_1' (set to the @id from previous cell)
main_record_set_id = record_set_ids[0]  # Adjust index if needed
df = dataframes[main_record_set_id]
print(f"Columns in `{main_record_set_id}`: {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's perform basic filtering, normalization, and groupby operations using numeric and categorical fields.

Replace `<numeric_field_id>` and `<group_field_id>` with the field `@id`s from above. For demonstration, we'll attempt to auto-select fields based on data types if present. Adjust variables for your analysis if a different choice is appropriate.

In [ ]:
# Automatically pick a numeric field and a group field for demonstration
numeric_field_id = None
group_field_id = None

# Try to auto-detect fields by inspecting dtype and name
for col in df.columns:
    if numeric_field_id is None and pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
    if group_field_id is None and (df[col].dtype == object) and ('id' in col.lower() or 'group' in col.lower() or 'category' in col.lower()):
        group_field_id = col

if numeric_field_id is None:
    print("No numeric field found. Please set `numeric_field_id` explicitly using a @id from above.")
else:
    print(f"Using numeric field: {numeric_field_id}")

if group_field_id is None and len(df.select_dtypes('object').columns) > 0:
    group_field_id = df.select_dtypes('object').columns[0]
    print(f"Defaulting group_field_id to: {group_field_id}")

# Filter by a threshold
if numeric_field_id:
    threshold = df[numeric_field_id].mean()  # Use mean as threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records in `{numeric_field_id}` > {threshold:.2f}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized `{numeric_field_id}` for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Groupby (if group field present)
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[[numeric_field_id]].mean()
        print(f"Grouped mean `{numeric_field_id}` by `{group_field_id}`:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn.

Again, adjust `numeric_field_id` and `group_field_id` as needed to fit available columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of `{numeric_field_id}`")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot by group
    if group_field_id in df:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
We loaded the FAIR$^2$ dataset using the Croissant standard with `mlcroissant`, explored record sets, and performed basic EDA and visualization with all fields referenced by their `@id` for programmatic clarity.

**Best Practices:**
- Always refer to dataset entities (`record_set`, `field`, etc.) by their `@id` attributes for reproducibility.
- Inspect metadata via the `dataset.metadata` object for additional dataset documentation, FAIR usage guidance, and schema-level details.

_You can now proceed further with domain-specific analysis, model training, and reproducible research using this FAIR dataset._